In [2]:
import pandas as pd

In [3]:
df_query_class_project_full = pd.read_csv('data/Queryclassproject_Full_2026-04-02.csv')
# df.to_excel('clt_pipe_HD_sales_raw.xlsx', index=False)
df_home_depot_sales_data = pd.read_csv('data/home_depot_sales_data.csv')

In [4]:
print(df_query_class_project_full.head())
print(df_query_class_project_full.count())

                      month                            sku name  sku nbr  \
0  Fiscal Period 12 of 2025  1-1/2" ABS DOUBLE SANI TEE HXHXHXH   134633   
1   Fiscal Period 3 of 2026  1-1/2" ABS DOUBLE SANI TEE HXHXHXH   134633   
2   Fiscal Period 2 of 2025  1-1/2" ABS DOUBLE SANI TEE HXHXHXH   134633   
3   Fiscal Period 1 of 2026  1-1/2" ABS DOUBLE SANI TEE HXHXHXH   134633   
4  Fiscal Period 10 of 2025  1-1/2" ABS DOUBLE SANI TEE HXHXHXH   134633   

  state territory code                 sub class                  region  \
0                   AK  026-001-005-ABS FITTINGS  0012-PACIFIC NORTHWEST   
1                   AK  026-001-005-ABS FITTINGS  0012-PACIFIC NORTHWEST   
2                   AK  026-001-005-ABS FITTINGS  0012-PACIFIC NORTHWEST   
3                   AK  026-001-005-ABS FITTINGS  0012-PACIFIC NORTHWEST   
4                   AK  026-001-005-ABS FITTINGS  0012-PACIFIC NORTHWEST   

   sales units  sales $  
0          5.0    62.93  
1         -1.0   -13.11  
2       

In [5]:
print(df_home_depot_sales_data.head())
print(df_home_depot_sales_data.count())

        month                 sku_name  sku_num             region  \
0  2024-04-01  CPVC-GLD-0.5-PIPE [2FT]   206041  PACIFIC_NORTHWEST   
1  2025-09-01  CPVC-GLD-0.5-PIPE [2FT]   206041  PACIFIC_NORTHWEST   
2  2025-02-01  CPVC-GLD-0.5-PIPE [2FT]   206041  PACIFIC_NORTHWEST   
3  2025-04-01  CPVC-GLD-0.5-PIPE [2FT]   206041  PACIFIC_NORTHWEST   
4  2024-12-01  CPVC-GLD-0.5-PIPE [2FT]   206041  PACIFIC_NORTHWEST   

   sales_units  
0     2.418678  
1    12.006578  
2     4.480735  
3     7.926598  
4     4.778459  
month          33522
sku_name       33522
sku_num        33522
region         33522
sales_units    33522
dtype: int64


In [6]:
df_query_class_project_full['sku nbr'].nunique()

1031

In [7]:
df_query_class_project_full.loc[df_query_class_project_full["sku nbr"] == 187976, "sales units"].sum()

np.float64(9640523.0)

In [8]:
sales_total = df_query_class_project_full["sales units"].sum()

df_sku_sales = (
    df_query_class_project_full
    .groupby("sku nbr", as_index=False)["sales units"]
    .sum()
    .rename(columns={"sales units": "sales_total"})
)

df_sku_sales = df_sku_sales.sort_values("sales_total", ascending=False).reset_index(drop=True)

df_sku_sales["sales_percent"] = df_sku_sales["sales_total"] / sales_total
df_sku_sales["cumulative_sales_percent"] = df_sku_sales["sales_percent"].cumsum()

df_top_95_sales = df_sku_sales[df_sku_sales["cumulative_sales_percent"] <= 0.95].copy()

if not df_sku_sales[df_sku_sales["cumulative_sales_percent"] > 0.95].empty:
    first_over_95 = df_sku_sales[df_sku_sales["cumulative_sales_percent"] > 0.95].head(1)
    df_top_95_sales = (
        pd.concat([df_top_95_sales, first_over_95])
        .drop_duplicates(subset=["sku nbr"])
        .reset_index(drop=True)
    )

df_top_95_sales

,sku nbr,sales_total,sales_percent,cumulative_sales_percent
0,187976,9640523.0,0.030232,0.030232
1,188077,7979098.0,0.025022,0.055253
2,188085,5953628.0,0.018670,0.073923
3,221646,5696715.0,0.017864,0.091787
4,187984,5333125.0,0.016724,0.108511
...,...,...,...,...
326,745274,143434.0,0.000450,0.948401
327,188816,141835.0,0.000445,0.948845
328,744917,139673.0,0.000438,0.949283
329,1003953659,132895.0,0.000417,0.949700


In [9]:
df_query_class_project_full_top_95 = df_query_class_project_full[
    df_query_class_project_full["sku nbr"].isin(df_top_95_sales["sku nbr"])
].copy()


In [10]:
df_query_class_project_full_top_95.count()

month                   773501
sku name                773501
sku nbr                 773501
state territory code    773501
sub class               773501
region                  773501
sales units             711094
sales $                 711094
dtype: int64

In [11]:
df_query_class_project_full_top_95.to_csv('data/clt_pipe_HD_sales_0_95_raw.csv', index=False)

In [12]:
df_query_class_project_full_top_95.head(1)

,month,sku name,sku nbr,state territory code,sub class,region,sales units,sales $
12,Fiscal Period 2 of 2026,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,5.0,24.8


In [13]:
df_clean = df_query_class_project_full_top_95.copy()
df_clean = df_clean.drop(columns=['sales $'])
df_clean['month_year'] = df_clean['month'].str[-4:]
df_clean['month_number'] = df_clean['month'].str.extract(r'Fiscal Period (\d+)').astype(int)
df_clean = df_clean.drop(columns=['month'])
df_clean['month'] = (df_clean['month_number'].astype(str) + '-' + pd.to_datetime(df_clean['month_year']).dt.strftime('%b-%Y'))
df_clean['month'] = pd.to_datetime('1-' + df_clean['month_number'].astype(str) + '-' + df_clean['month_year'].astype(str), format='%d-%m-%Y')
df_clean = df_clean.dropna(subset=['sales units'])
df_clean.head(5)

,sku name,sku nbr,state territory code,sub class,region,sales units,month_year,month_number,month
12,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,5.0,2026,2,2026-02-01
13,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2023,12,2023-12-01
14,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2025,7,2025-07-01
15,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,7.0,2025,4,2025-04-01
16,"3/4"" PVC PTRAP SPGXSPG",142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2025,8,2025-08-01


In [14]:
df_clean = df_clean.assign(sku=df_clean['sku name'].str.replace(r"[\",\-']", "", regex=True))
df_clean = df_clean.drop(columns=['sku name'])
df_clean['sku'] = df_clean['sku'].str.replace(' ', '_')
df_clean = df_clean.rename(columns={'sku': 'sku name'})
df_clean.head()

,sku nbr,state territory code,sub class,region,sales units,month_year,month_number,month,sku name
12,142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,5.0,2026,2,2026-02-01,3/4_PVC_PTRAP_SPGXSPG
13,142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2023,12,2023-12-01,3/4_PVC_PTRAP_SPGXSPG
14,142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2025,7,2025-07-01,3/4_PVC_PTRAP_SPGXSPG
15,142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,7.0,2025,4,2025-04-01,3/4_PVC_PTRAP_SPGXSPG
16,142559,AK,026-001-003-PVC FITTINGS,0012-PACIFIC NORTHWEST,4.0,2025,8,2025-08-01,3/4_PVC_PTRAP_SPGXSPG


In [15]:
df_clean.count()

sku nbr                 711094
state territory code    711094
sub class               711094
region                  711094
sales units             711094
month_year              711094
month_number            711094
month                   711094
sku name                711094
dtype: int64

In [16]:
df_transform = df_clean.copy()
df_transform = df_transform.drop(columns=['state territory code','region','month_number','month_year'])
df_transform.head(1)

,sku nbr,sub class,sales units,month,sku name
12,142559,026-001-003-PVC FITTINGS,5.0,2026-02-01,3/4_PVC_PTRAP_SPGXSPG


In [17]:
df_transform["sales units"] = df_transform.groupby(["sku nbr", "month"])["sales units"].transform("sum")
df_transform = df_transform.drop_duplicates(subset=["sku nbr", "month"]).reset_index(drop=True)
df_transform.head(1)

,sku nbr,sub class,sales units,month,sku name
0,142559,026-001-003-PVC FITTINGS,11747.0,2026-02-01,3/4_PVC_PTRAP_SPGXSPG


In [18]:
df_transform.count()

sku nbr        12909
sub class      12909
sales units    12909
month          12909
sku name       12909
dtype: int64

In [19]:
df_transform = df_transform[df_transform["month"] != "2026-03-01"].copy()
print(df_transform.count())
print(df_transform.head(1))

sku nbr        12578
sub class      12578
sales units    12578
month          12578
sku name       12578
dtype: int64
   sku nbr                 sub class  sales units      month  \
0   142559  026-001-003-PVC FITTINGS      11747.0 2026-02-01   

                sku name  
0  3/4_PVC_PTRAP_SPGXSPG  


In [20]:
df_transform['sku nbr'].nunique()

331

In [43]:
df_transform.to_csv('data/clt_pipe_HD_sales_0_95_transformed.csv', index=False)